# Identifying student outcomes in messages: a functional linguistic approach  
<br> <b>Jon Ball</b> 
<br> 4/1/2022

<b>Research Question:</b>
<br>
How can we identify information about student outcomes in the announcements and check-in messages teachers send to students' families?
<br>
<br>
<b>Goal:</b>
<br>
Using regular expressions that filter text messages by <b>part of speech</b> (form) and <b>vocabulary</b> (content), identify information about student outcomes. Classify text messages based on whether they contain information about student outcomes, using linguistic criteria. These criteria are:
<br>
<br> 1. (part of speech) superlative and comparative adjectives and adverbs
<br> 2. (part of speech) predeterminers
<br> 3. (part of speech) present participle, or verbs ending in "-ing"
<br> 4. (part of speech) possesive pronouns and endings
<br> 5. (vocabulary) "improvement" words
<br> 6. (vocabulary) "difficulty" words

Import packages:

In [ ]:
import numpy as np
import pandas as pd

Load 99,968 non-null TP messages, sent by **teachers**:

In [ ]:
tpm = pd.read_csv('data/rand100k_teachers.csv')
tpm = tpm[tpm['TEXTENGLISH'].notnull()]
tpm.info()

## Regular expressions for identifying student performance information
<br>
<b>Objective:</b> 
<br>
Identify messages about student outcomes using regexes that capture parts of speech and vocabulary.

### Parts of speech: comparisons, quantifiers, ongoing actions, and possessives

#### Superlative and comparative adverbs and adjectives are used to compare present and past performance, expectations and outcomes, as well as test scores and grades: <br>

In [ ]:
tpm['comparison'] = tpm['TEXTENGLISH'].str.contains(r"\b\w{3,}er\b|\bmore\b|\bworse\b|\bless\b|\b\w{3,}est\b|\bmost\b|\bbest\b\|\bworst\b|\bleast\b",
                                                    case=False,
                                                    regex=True)
for item in tpm[tpm['comparison']==True].sample(n=2, random_state=1234)['TEXTENGLISH'].tolist():
    print(item, '\n')

#### Predeterminers quantify the amount or extent of classes, assignments, or schoolwork:

In [ ]:
tpm['quantifier'] = tpm['TEXTENGLISH'].str.contains(r"\ball\b|\banother\b|\bany\b|\bboth\b|\beach\b|\beither\b|\benough\b|\bevery\b|\bfew\b|\blast\b|\bleast\b|\blittle\b|\bmany\b|\bmuch\b|\bneither\b|\bnext\b|\bnone\b|\bnothing\b|\bone\b|\bseveral\b|\bsome\b|\bsufficient\b|\bthree\b|\btwo\b|\bvarious\b|\bzero\b",
                                                    case=True,
                                                    regex=True)
for item in tpm[tpm['quantifier']==True].sample(n=2, random_state=1234)['TEXTENGLISH'].tolist():
    print(item, '\n')

#### Present participle indicates present or ongoing actions, and descriptions of ongoing performance or behavior:

In [ ]:
tpm['ongoing_action'] = tpm['TEXTENGLISH'].str.contains(r"(is|\'s|are|\'re|not|\'nt|been)\s+\w*ing\b",
                                                        case=False,
                                                        regex=True)
for item in tpm[tpm['ongoing_action']==True].sample(n=2, random_state=1234)['TEXTENGLISH'].tolist():
    print(item, '\n')

#### Possesive pronouns and endings indicate ownership and responsibility:

In [ ]:
tpm['possessive'] = tpm['TEXTENGLISH'].str.contains(r"\bhis\b|\bhers?\b|theirs?\b|'s\b",
                                                    case=False,
                                                    regex=True)
for item in tpm[tpm['possessive']==True].sample(n=2, random_state=1234)['TEXTENGLISH'].tolist():
    print(item, '\n')

<b>Proportion of messages in the 99.6k sample that are tagged "academic" using the REGEX approach:</b>

In [ ]:
acad_n = tpm[(tpm['comparison'] == True) & 
             (tpm['quantifier'] == True) & 
             (tpm['ongoing_action'] == True) & 
             (tpm['possessive'] == True)].shape
print("{} messages determined to be student outcome-related by the above regexes.".format(acad_n[0]))
print("Proportion of messages determined to be 'academic' by the above regexes: {}".format(acad_n[0] / tpm.shape[0]))

### Measuring precision of REGEX approach with a sample of 256 teacher messages:

Pull a subsample of 256 messages as a validation set:

In [ ]:
sample = tpm[(tpm['comparison'] == True) & 
             (tpm['quantifier'] == True) & 
             (tpm['ongoing_action'] == True) & 
             (tpm['possessive'] == True)].sample(n=256, random_state=1234)

Print the messages contained in the sample for labeling:

In [ ]:
for idx, message in enumerate(sample['TEXTENGLISH'].tolist()):
    print("{}: {}".format(idx+1, message))
    print('--\n')

Create a label vector of shape (256, 1) corresponding to REGEX-filtered messages:

In [ ]:
#1 is academic, positive
#0 is non-academic
#-1 is academic, negative

#16 rows of 16 corresponding to 
sample_classes = np.array(
    [
        -1, 0, -1, 1, 1, -1, -1, -1, -1, 1, 1, -1, -1, 0, 1, -1,
        -1, -1, 0, 0, 0, 0, -1, 0, -1, -1, -1, 1, 0, -1, 1, -1,
        -1, -1, -1, -1, -1, 0, -1, -1, 0, -1, -1, -1, 0, -1, 0, -1,
        0, -1, -1, -1, 0, -1, 0, 0, -1, 0, 1, 0, 0, 0, -1, 0,
        -1, -1, 1, -1, -1, 0, -1, 0, 0, 0, 0, 0, -1, -1, 0, -1,
        0, 0, -1, -1, -1, 0, -1, -1, 0, 0, 0, -1, 0, 0, -1, -1,
        -1, -1, -1, -1, -1, 0, 0, 0, 0, -1, 0, -1, 0, 0, 0, 1,
        -1, -1, -1, 0, -1, 1, -1, -1, -1, -1, -1, -1, -1, -1, 0, -1,
        -1, 1, 0, -1, 1, 0, 0, 0, -1, 0, 0, 1, 0, -1, -1, -1,
        0, 1, -1, -1, -1, -1, 1, -1, -1, -1, -1, 0, -1, -1, 0, -1, 
        0, -1, 0, 0, 0, -1, -1, 0, 1, 0, 0, 0, -1, 0, 1, -1,
        0, 1, -1, 0, 0, -1, 0, 0, 0, 0, 1, 0, -1, -1, -1, -1,
        0, 0, -1, 0, -1, 1, -1, -1, 0, -1, -1, 1, 0, -1, 0, -1,
        -1, 1, 0, 0, -1, -1, 0, 1, -1, 0, -1, -1, -1, 1, -1, -1,
        0, 0, 0, -1, -1, 1, -1, -1, 1, -1, 0, 0, 0, -1, -1, -1,
        0, -1, 0, -1, -1, 1, 1, 0, -1, -1, 0, 0, 0, -1, -1, -1
    ]
)
print('Correctly identified {} messages as relating to academic outcomes. ({})'.format(np.count_nonzero(sample_classes), np.count_nonzero(sample_classes) / 256))

Compare to a random sample of 256 messages:

In [ ]:
for idx, message in enumerate(tpm.sample(n=256, random_state=42)['TEXTENGLISH'].tolist()):
    print("{}: {}".format(idx+1, message))
    print('--\n')

Create a label vector of shape (256, 1) for the random messages:

In [ ]:
#1 is academic, positive
#0 is non-academic
#-1 is academic, negative

#16 rows of 16 corresponding to a (256, 1) 1-D array of classes for each message
random_classes = np.array([0, 1, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0,
                           0, 1, 0, 0, 0, 1, 0, 0, -1, 0, 0, 0, 0, 0, 0, 0,
                           0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, -1, 1, 0, 0,
                           0, 1, 0, 0, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
                           0, 0, -1, 0, 0, -1, 0, 0, 0, -1, 0, 0, -1, 0, 0, 0,
                           0, 0, 0, -1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, -1, 0,
                           0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
                           0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1,
                           -1, 0, 0, 0, 0, 0, -1, 0, 0, -1, -1, -1, 0, 1, 0, 0,
                           0, 0, -1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
                           -1, 0, 0, 0, -1, 0, 0, -1, 0, 0, 0, 0, 0, 0, 0, 0,
                           0, 0, 0, 1, 0, -1, 0, 1, 0, 0, 0, 0, 0, -1, 0, 0,
                           -1, 0, 0, 0, 0, 0, -1, 0, 0, 0, 0, -1, 0, 0, 0, 0, 
                           0, 1, -1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, -1, 0, 0,
                           0, 0, 0, -1, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
                           0, 0, 0, -1, 0, 0, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0])
print('{} random messages indicate academic performance. {}%'.format(np.count_nonzero(random_classes), np.count_nonzero(random_classes) / 256))

# Feature vector approach

Load spacy model for part of speech tagging:

In [ ]:
#!pip install spacy spacytextblob
#!python -m spacy download en_core_web_lg

In [ ]:
import spacy
from spacytextblob.spacytextblob import SpacyTextBlob

In [ ]:
nlp = spacy.load('en_core_web_lg')
nlp.add_pipe('spacytextblob')

Generate a list of binary feature vectors for each message:

In [ ]:
list_of_vecs = []

for message in nlp.pipe(tpm['TEXTENGLISH'].tolist()):
    
    new_vec = np.zeros(4)
    
    for ent in message: #for all entities in the message
            
        if ent.tag_ == 'RBS' or 'JJS' or 'RBR' or 'JJR': 
            # RBS : adverb, superlative - quickest
            # JJS : adjective, superlative - hardest
            # RBR : adverb, comparative - better
            # JJR : adjective, comparative - easier
            new_vec[0] = 1
            
        if ent.tag_ == 'PDT': # PDT : predeterminer - all, most, any
            new_vec[1] = 1

        if ent.tag_ == 'POS' or 'PRP$': 
            # POS : possessive ending - student's
            # PRP$ : possessive pronoun - her/his/their
            new_vec[2] = 1
            
        if ent.tag_ == 'VBG' or 'VBZ': 
            # VBG : verb, gerund, or present participle - failing
            # VBZ : verb, present tense 3rd person - tries
            new_vec[3] = 1
            
    list_of_vecs.append(new_vec)

In [ ]:
tpm['features'] = list_of_vecs

In [ ]:
n_features = []
for vec in list_of_vecs:
    n_features.append(np.sum(vec))

In [ ]:
tpm['n_features'] = n_features

In [ ]:
n2tpm = tpm[tpm['n_features'] > 2]
n2tpm.shape

In [ ]:
for message in n2tpm['TEXTENGLISH'].tolist():
    print(message)
    print('--\n')

Classify messages by positive and negative sentiment:

In [ ]:
sentiment_list = []
for message in nlp.pipe(tpm['TEXTENGLISH'].tolist()):
    if message._.blob.polarity > 0:
        sentiment_list.append(1)
    else:
        sentiment_list.append(0)
print(len(sentiment_list))

In [ ]:
tpm['SENTIMENT'] = sentiment_list